# Altiostar Tokyo MRO - Google Colab Training Notebook

This notebook enables fast training of the Reinforcement Learning MRO optimization models for Altiostar using Google Colab's GPU acceleration (CUDA).

### Setup Instructions:
1. In the top menu, go to **Runtime** > **Change runtime type**.
2. Select **T4 GPU** (or any available GPU) and click **Save**.
3. Run the cells below sequentially.

## 1. Clone Repository & Setup Environment

In [ ]:
# Clone the repository (replace with your fork/branch if needed)
!git clone https://github.com/LifeAtlas/altiostar-tokyo-mro.git
%cd altiostar-tokyo-mro

# Checkout Shourya's seeded runs feature branch
!git checkout feature/shourya-seeded-runs

In [ ]:
# Install all requirements and packages needed for ONNX export
!pip install -r requirements.txt
!pip install onnx

## 2. Enable GPU Execution & Generate Data

In [ ]:
# Dynamically edit run_experiment.py to use GPU runtime (cuda) instead of CPU
!sed -i 's/device="cpu"/device="cuda"/g' run_experiment.py

# Generate the relation-level PM dataset from cell-level PM data
!python scripts/generate_relation_pm.py

## 3. Run the 50K Seeded Experiments

In [ ]:
# Train all 5 seeds (seeds 1 to 5) for 50,000 steps each
# This will run significantly faster on the GPU (~3 min per run)
!python scripts/run_seeded_experiments.py --timesteps 50000 --eval-episodes 5

## 4. Validate Ship Gate & Export ONNX Models

In [ ]:
# Verify that the trained seed 1 results meet a relaxed gate check (e.g., >75% HO Success)
!python src/pipeline/ship_gate.py --results results/seeded_runs/experiment_v2_baseline_seed_1.json --ho-success-min 75.0

In [ ]:
# Export the trained PyTorch policies to ONNX format for deployment
!python src/pipeline/export_onnx.py --model checkpoints/seeded_runs/ppo_v2_baseline_seed_1.zip --output checkpoints/seeded_runs/ppo_v2_baseline_seed_1.onnx

## 5. Compress and Download Results

In [ ]:
# Zip the generated results and model checkpoints for easy download
!zip -r mro_colab_results.zip results/ checkpoints/
print("Zipped successfully! You can download 'mro_colab_results.zip' from the files tab on the left.")